# 06 — Retrieval, neighborhoods, and a link-prediction smoke

The final lesson combines three workflows on deterministic fixture data: embedding retrieval, relation-neighborhood inspection, and a tiny PyG link-prediction run. The emphasis is experimental design—split leakage, unknown negatives, metrics, and error analysis—not benchmark performance or therapeutic recommendation.


In [ ]:
from __future__ import annotations
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from manage_db.public_notebooks import (
    PUBLIC_KG_ROOT,
    build_public_fixture,
    parquet_catalog,
    read_bounded_parquet,
)

MODE = os.environ.get("JOUVENCE_DATA_MODE", "fixture")
BILLING_PROJECT = os.environ.get("JOUVENCE_BILLING_PROJECT")
CACHE = REPO_ROOT / "artifacts" / "cache" / "public-notebooks"
CACHE.mkdir(parents=True, exist_ok=True)
KG_ROOT = build_public_fixture(CACHE / "kg-fixture") if MODE == "fixture" else PUBLIC_KG_ROOT
print({"mode": MODE, "kg_root": str(KG_ROOT), "bounded": True})

try:
    import torch
    import torch_geometric
except ImportError as exc:
    raise RuntimeError("Install the notebook GNN environment with: uv sync --group notebooks --group gnn") from exc


## Define the scientific task before selecting data

A useful workflow names the target relation, prediction unit, intended generalization regime, allowed input features, temporal/source cutoff, and evaluation population. Storage convenience should not decide the split.


In [ ]:
task_card = {
    "target_relation": "disease_associated_gene",
    "prediction_unit": "typed gene–disease pair",
    "generalization": "software smoke on fixture only",
    "input_policy": "fixture features with declared fallback",
    "forbidden_claim": "clinical target recommendation",
}
print(json.dumps(task_card, indent=2))


### Interpretation

A random edge split often leaves the same entities, near-duplicate evidence, and ontology neighbors in train and test. Zero-shot disease or prospective evaluation requires entity-, time-, or source-aware splitting.


In [ ]:
split_options = pd.DataFrame([
    ("random edge", "pipeline smoke", "shared entities/evidence leakage"),
    ("held-out disease", "zero-shot disease", "ontology/text leakage"),
    ("temporal", "prospective claim", "post-cutoff metadata leakage"),
    ("source-held-out", "cross-source robustness", "duplicate assertions across sources"),
], columns=["split", "question", "principal risk"])
display(split_options)


### Checkpoint

Choose a split that matches the deployment claim, then audit every embedding and metadata field against that split.


## Retrieve representation neighbors for inspection

Retrieval surfaces entities whose vectors are close under one accepted modality. It is useful for candidate inspection and error analysis, but it is not itself link prediction and does not establish a biological relation.


In [ ]:
from manage_db.public_notebooks import nearest_embeddings
embedding_uri = Path(KG_ROOT) / "features" / "embeddings" / "text" / "fixture.parquet"
neighbors = nearest_embeddings(embedding_uri, "ENSG00000012048", limit=5)
display(neighbors)


### Interpretation

The neighbor list inherits the synthetic fixture encoder. Proximity is not functional equivalence, interaction, causality, or therapeutic interchangeability; compare modalities and inspect source payloads before interpretation.


In [ ]:
print({"query": "ENSG00000012048", "retrieved": len(neighbors), "metric": "cosine", "modality": "synthetic text fixture"})


### Checkpoint

Use Notebook 02's norm, distribution, coverage, and projection diagnostics before relying on a retrieval threshold.


## Inspect typed neighborhoods without inflating evidence

A relation neighborhood summarizes represented adjacency. Degree can identify hubs or sampling artifacts, while evidence must be inspected separately so many source records do not become duplicate graph edges.


In [ ]:
edge_uri = Path(KG_ROOT) / "edges" / "disease_associated_gene.parquet"
edge_sample = read_bounded_parquet(edge_uri, limit=100)
degree = edge_sample.groupby("x_id").size().rename("sampled_disease_degree").sort_values(ascending=False)
display(degree)


### Interpretation

Degree in this fixture is a property of the tiny selected graph. In live bounded prefixes it is not global degree, and in either mode it does not measure causal importance or targetability.


In [ ]:
evidence_uri = Path(KG_ROOT) / "evidence" / "disease_associated_gene.parquet"
evidence_sample = read_bounded_parquet(evidence_uri, limit=100)
evidence_counts = evidence_sample.groupby("x_id").size().rename("sampled_evidence_rows")
display(pd.concat([degree, evidence_counts], axis=1).fillna(0))


### Checkpoint

Keep adjacency degree and evidence multiplicity in separate columns and interpretations. Neither is a clinical ranking.


## Build the sample and run deterministic link prediction

The repository smoke creates a bounded PyG export, deterministic train/validation/test partitions, sampled unknown pairs, and a tiny model run. Fixed seeds make software behavior repeatable; they do not make fixture metrics scientifically meaningful.


In [ ]:
from manage_db.public_notebooks import build_sampled_pyg, run_sampled_ml
if MODE != "fixture":
    raise RuntimeError("Train only on an explicitly staged bounded subset with a reviewed leakage policy")
PYG_ROOT = CACHE / "pyg-ml-course-sample"
build_sampled_pyg(KG_ROOT, PYG_ROOT, max_nodes_per_type=100, max_edges_per_relation=200)
smoke = run_sampled_ml(PYG_ROOT, seed=13)
print({"status": smoke["status"], "split_counts": smoke["split_counts"]})


### Interpretation

Negative samples are unobserved or unknown pairs under the selected graph, not proven biological negatives. False-negative contamination is especially likely in incomplete biomedical KGs.


In [ ]:
metrics = pd.Series(smoke["metrics"], name="value").to_frame()
display(metrics)
print(json.dumps(smoke["validation"], indent=2))


### Checkpoint

A two-epoch fixture result is only a runtime smoke. Do not compare its values with publications, call it model validation, or use it for drug-repurposing decisions.


## Match metrics to imbalance and decision costs

AUROC can look optimistic under extreme class imbalance; average precision exposes positive retrieval quality; ranking metrics such as Hits@K and MRR require a documented candidate set and filtering policy; calibration matters when scores are interpreted probabilistically.


In [ ]:
metric_guide = pd.DataFrame([
    ("AUROC", "pair ordering", "can hide low precision under imbalance"),
    ("average precision", "positive retrieval", "depends on prevalence/candidate set"),
    ("Hits@K / MRR", "ranking", "requires filtered candidate protocol"),
    ("calibration", "probability-like decisions", "needs representative held-out data"),
], columns=["metric", "use", "caveat"])
display(metric_guide)


### Interpretation

No single metric validates biomedical utility. Report confidence intervals across seeds or splits, baselines, candidate denominators, and decision-relevant error costs for any real experiment.


In [ ]:
reported_boundary = {
    "epochs": 2,
    "fixture": True,
    "benchmark": False,
    "biological_validation": False,
    "therapeutic_recommendation": False,
}
print(reported_boundary)


### Checkpoint

For this course, the only accepted conclusion is that the bounded code path executes and returns finite structured outputs.


## Perform error analysis and close the course honestly

Error analysis joins high-scoring mistakes back to stable IDs, node labels, relation evidence, feature coverage, and split provenance. Look for hubs, duplicates, source overlap, missing features, ontology leakage, and false-negative candidates before changing the model.


In [ ]:
error_checklist = [
    "map tensor positions back to stable IDs",
    "inspect source evidence and release cutoff",
    "check feature coverage and fallback masks",
    "look for duplicate or near-duplicate entities",
    "review unknown negatives as possible false negatives",
    "stratify by node degree and modality coverage",
]
print("Error-analysis checklist:\n- " + "\n- ".join(error_checklist))


### Interpretation

What this means: retrieval, neighborhood, export, splitting, and a tiny training loop are executable together. What this does not mean: the model generalizes, full-KG training is done, scores are calibrated, or any candidate is effective or safe.


In [ ]:
course_summary = {
    "notebooks_completed": 6,
    "data_plane": "fixture by default; bounded named live reads only",
    "model_status": "bounded training smoke passed",
    "production_full_done": False,
    "clinical_use": False,
}
print(json.dumps(course_summary, indent=2))


### Checkpoint

Return to Notebook 01 for access and data identity, Notebook 02 for embedding diagnostics, Notebook 03 for evidence, Notebook 04 for LaminDB, or Notebook 05 for tensor contracts. Those boundaries are part of the result, not disclaimers to omit.
